/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1EVVPtk2XMHbGrUEumFHWcwJQYmpq-l6V
From (redirected): https://drive.google.com/uc?id=1EVVPtk2XMHbGrUEumFHWcwJQYmpq-l6V&confirm=t&uuid=ca6fccf0-88ab-45d7-9092-a42fda04a1cd
To: /content/Training_database_float16.zip
100% 5.53G/5.53G [01:55<00:00, 47.6MB/s]


In [3]:
import os
import zipfile
import random
import pandas as pd
import numpy as np
from tqdm import tqdm
from skimage.transform import resize
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')

# Force use of the second GPU (usually RTX)
tf.config.set_visible_devices(gpus[1], 'GPU')

print("Using:", tf.config.list_physical_devices('GPU'))

In [4]:
zip_path = "/content/Training_database_float16.zip"
real_data_zip_path = "/content/drive/MyDrive/MD4 2025-2026/Computer vision/real_data.zip"
extraction_path = "/content/drive/MyDrive/skipper_data"
data_path="Training_database_float16/Training_database_float16"
csv_path = "pipe_detection_label.csv"

In [6]:
os.makedirs(extraction_path, exist_ok=True)
with zipfile.ZipFile(zip_path, "r") as zip_object:
  zip_object.extractall(extraction_path)

In [5]:
SEED = 42
SPLIT = 0.2

In [6]:
def load_dataset(data_path, full_image=True):


  df_labels = pd.read_csv(csv_path, sep=";")

  label_dict = dict(zip(df_labels["field_file"], df_labels["label"]))

  file_names = []
  images = []
  labels = []

  files = os.listdir(data_path)
  random.shuffle(files)

  for file in tqdm(files):
    if file.endswith(".npz") and file in label_dict:
      path = os.path.join(data_path, file)

      data = np.load(path)
      array = data["data"]
      if not full_image:
        array = array[:,:, 2]# Bz
        array = resize(array, (224, 224, 1), anti_aliasing=True)
      else:
        array = resize(array, (224, 224, 4), anti_aliasing=True)

      array = np.nan_to_num(array)
      if np.nanmax(array) != 0:
        array /= np.nanmax(array)

      images.append(array)
      labels.append(label_dict[file])
      file_names.append(file)


  X = np.array(images)
  y = np.array(labels)
  return X, y, file_names

In [7]:
X, Y, file_names = load_dataset(data_path= data_path)

100%|██████████| 2852/2852 [07:16<00:00,  6.54it/s]


In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=SPLIT, random_state=42, stratify=Y)

In [ ]:
from tensorflow import keras

In [11]:
def init_pipe_existance_classifier_model(full_image=True):
  input_shape = (224, 224, 4) if full_image else (224, 224,1)
  pipe_existance_classifier_model = keras.Sequential(
      [
      keras.Input(shape=input_shape),
      #encodeur
      keras.layers.Conv2D(32, kernel_size=(3, 3), strides=(1,1), activation="relu"),
      keras.layers.MaxPool2D(pool_size=(2,2)),

      keras.layers.Conv2D(64, kernel_size=(3, 3), strides=(1,1), activation="relu"),
      keras.layers.MaxPool2D(pool_size=(2,2)),

      keras.layers.Conv2D(128, kernel_size=(3, 3), strides=(1,1), activation="relu"),
      keras.layers.MaxPool2D(pool_size=(2,2)),

      # interconnexion encodeur <-> classifieur
      keras.layers.Flatten(),

      # classifieur
      keras.layers.Dense(100),
      keras.layers.Dense(1, activation="sigmoid")
      ]
  )
  return pipe_existance_classifier_model

In [12]:
pipe_existance_classifier_model = init_pipe_existance_classifier_model()

In [13]:
pipe_existance_classifier_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │         1,184 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 100)            │     8,652,900 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           101 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,746,537 (33.37 MB)

 Trainable params: 8,746,537 (33.37 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
pipe_existance_classifier_model.compile(loss="binary_crossentropy", optimizer=keras.optimizers.Adam(learning_rate=10**(-4)), metrics=["accuracy"])

In [15]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True,
)

In [16]:
pipe_existance_classifier_model.fit(X_train, y_train, validation_split=0.2, batch_size=16, epochs=10_000, callbacks=[early_stopping])

Epoch 1/10000
114/114 ━━━━━━━━━━━━━━━━━━━━ 52s 446ms/step - accuracy: 0.9558 - loss: 0.1099 - val_accuracy: 0.9692 - val_loss: 0.0566
Epoch 2/10000
114/114 ━━━━━━━━━━━━━━━━━━━━ 51s 447ms/step - accuracy: 0.9774 - loss: 0.0362 - val_accuracy: 0.9780 - val_loss: 0.0496
Epoch 3/10000
114/114 ━━━━━━━━━━━━━━━━━━━━ 51s 444ms/step - accuracy: 0.9779 - loss: 0.0298 - val_accuracy: 0.9802 - val_loss: 0.0368
Epoch 4/10000
114/114 ━━━━━━━━━━━━━━━━━━━━ 50s 437ms/step - accuracy: 0.9796 - loss: 0.0276 - val_accuracy: 0.9802 - val_loss: 0.0367
Epoch 5/10000
114/114 ━━━━━━━━━━━━━━━━━━━━ 51s 444ms/step - accuracy: 0.9785 - loss: 0.0270 - val_accuracy: 0.9758 - val_loss: 0.0364
Epoch 6/10000
114/114 ━━━━━━━━━━━━━━━━━━━━ 50s 437ms/step - accuracy: 0.9823 - loss: 0.0264 - val_accuracy: 0.9714 - val_loss: 0.0403
Epoch 7/10000
114/114 ━━━━━━━━━━━━━━━━━━━━ 51s 446ms/step - accuracy: 0.9818 - loss: 0.0262 - val_accuracy: 0.9802 - val_loss: 0.0405
Epoch 8/10000
114/114 ━━━━━━━━━━━━━━━━━━━━ 50s 439ms/step - ac

#test model


In [17]:
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [18]:
y_pred_probs = pipe_existance_classifier_model.predict(X_test)

18/18 ━━━━━━━━━━━━━━━━━━━━ 4s 231ms/step


In [26]:
y_pred_probs = pipe_existance_classifier_model.predict(X_test)

y_pred = (y_pred_probs>0.03).astype(int).flatten()


print(f"Accuracy:{accuracy_score(y_test, y_pred)}")
print(f"Classification Report:\n{classification_report(y_test, y_pred)}")
print(f"\nConfusion Matrix:\n{confusion_matrix(y_test, y_pred)}")


18/18 ━━━━━━━━━━━━━━━━━━━━ 4s 205ms/step
Accuracy:0.9911816578483245
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.98      0.99       227
           1       0.99      1.00      0.99       340

    accuracy                           0.99       567
   macro avg       0.99      0.99      0.99       567
weighted avg       0.99      0.99      0.99       567


Confusion Matrix:
[[222   5]
 [  0 340]]
